In [ ]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
from musicdb import PanDBIO
from gate import IOStore, IDStore
from match import MatchDBDataIO, AlbumReq, MatchString, MatchCounts, MatchDBCounts, PanDBMulti
from uuid import uuid4
from pandas import Series

# Manual Missing

In [ ]:
mdbc = MatchDBCounts(db="RateYourMusic", verbose=True)

# Multi Matches

In [ ]:
pdbio = PanDBIO()
ids   = IDStore()
ios   = IOStore()
pdbm  = PanDBMulti()
multiMatchData = pdbm.get()

In [ ]:
from master import MasterDBs
from pandas import concat
dbs  = MasterDBs().getDBs()
base = "RateYourMusic"
for artistID,artistIDData in multiMatchData.items():
    mData = artistIDData['Rows'].T
    mData["Match"] = artistIDData['Data']
    mData = mData.T
    mdbs  = mData[dbs].count()
    mdbs  = mdbs[mdbs > 0].index
    data  = mData[["ArtistName"]+list(mdbs)]
    print("\n")
    print("="*200)
    print("="*200)
    print(data.to_string())
    print(f"pdbio.setrymid('', '{artistID}')")

In [ ]:
pdbio.getMMEByArtist(["Los Cinco Latinos", "Los cinco latinos", "Los  Cinco Latinos"])

In [ ]:
pdbio.setspotid("aedcce3b97f2xx1", "5T5Xw3jmM98NH8KMFB6qrX")
pdbio.setgenid("aedcce3b97f2xx1", "459736")
pdbio.setrymid("aedcce3b97f2xx1", "185521")
pdbio.dropRow("aedcce3b97f2xx2")
pdbio.saveData()

In [ ]:
pdbio.setrymid("80f170eac1b8xx1", "[Artist233820]")
pdbio.newArtist("Frequent Flyers", MusicBrainz="321788190879519973541587161950552244620", RateYourMusic="233839", Discogs="3799803")

In [ ]:
ids.get("MusicBrainz", "https://musicbrainz.org/artist/a0f60c2c-673f-4c3e-85bb-c2048ba6f48d")

In [ ]:
pdbio.setrymid("ssssssssXXX0044471XXX003", "[Artist17710]")
pdbio.setmburl("ssssssssXXX0044471XXX003", "https://musicbrainz.org/artist/2fc0a8ab-acd5-4d73-bfd7-cd9a275ee195https://musicbrainz.org/artist/362b8733-7ef7-4fc7-a264-7fdf54ea4f88")

In [ ]:
pdbio.dropRow('eeeeeeeeXXX0011795XXX01')
pdbio.dropRow('eeeeeeeeXXX0011795XXX02')
pdbio.dropRow('e6d664b7-5b9f-4251-94cb-a2693b9ef2cf')
pdbio.newArtist("Embee", Discogs="173406", MusicBrainz="63553473372150312547850760189973940274", RateYourMusic='1058565', Spotify='2Wqm0Ny4QfBTGGW2bTwDJi', Beatport='24312', Traxsource='181855') # Sweden
pdbio.newArtist("Embee", Discogs="1183", MusicBrainz="292694041070512234855160648956448643650", RateYourMusic='631587', AllMusic='0000151674') # UK

In [ ]:
pdbio.saveData()

In [ ]:
mdbio = ios.get("RateYourMusic")
mdbio

In [ ]:
mdbc = MatchDBCounts("RateYourMusic")
unMatched = mdbc.getUnmatched()

In [ ]:
unMatched[unMatched["ArtistName"] == "Phil Austin"]

In [ ]:
pdbio.getMMEByArtist("Phil Austin")

In [ ]:
pdbio.saveData()

In [ ]:
ids = IDStore()
ids.getmbid("https://musicbrainz.org/artist/4cf77ca1-09cb-4e74-b429-9c6601df1f3f")

In [ ]:
for artistID,row in mdbc.getUnmatched().head(40).tail(20).iterrows():
    print("pdbio.newArtist('{0}', Genius='{1}', Spotify='')".format(row["ArtistName"], artistID))

# LastFM <=> MusicBrainz

In [ ]:
mdbio = gate.getIO("LastFM")
mbrainz = gate.getIO("MusicBrainz")

In [ ]:
searchData = mdbio.data.getSearchArtistData()
searchData['id'] = searchData['url'].apply(mdbio.getdbid)
searchData['MB'] = searchData['mbid'].apply(mbrainz.getdbid)

In [ ]:
from pandas import Series
mbidmapData = searchData[searchData["MB"].notna() & searchData["id"].notna()].drop_duplicates(subset=["mbid","MB"])
mbidMap = Series(mbidmapData['id'].values, index=mbidmapData["MB"].values)

In [ ]:
mmeDF["LastFMNew"] = mmeDF["MusicBrainz"].map(mbidmap)

In [ ]:
mmeDF = mmeDF.drop(["LastFM"], axis=1)
mmeDF = mmeDF.rename(columns={"LastFMNew": "LastFM"})

# SetListFM <=> MusicBrainz

In [ ]:
def mergeSetListFM(row, slfmMap):
    retval = row["SetListFM"] if isinstance(row["SetListFM"],str) else slfmMap.get(row["MusicBrainz"], None)
    return retval

ios     = IOStore()
ids     = IDStore()
mio     = ios.get("SetListFM")
useWebData = True
if useWebData is True:
    saData  = mio.data.getSearchWebArtistData().reset_index().rename(columns={"index": "id"})
    saData.index = saData["MBID"].map(ids.getmbid)
    saData.index.name = ""
else:
    mio.data.getSearchArtistData()
    saData['MB'] = saData['mbid'].apply(ids.getmbid)
slfmMap = saData["id"]
print("Found {0} SetListFM <=> MusicBrainz Map Entries".format(len(slfmMap)))

pdbio = PanDBIO()
mmeDF = pdbio.getData()
print("Got PanDB Data")

In [ ]:
print("Mapping MusicBrainz <=> SetListFM", end=" ... ")
mmeDF["SetListFMNew"] = mmeDF["MusicBrainz"].map(slfmMap.get)
print("Done")

In [ ]:
mmeDF

In [ ]:
mmeDF["SetListFMNew"] = mmeDF[["MusicBrainz", "SetListFM"]].apply(mergeSetListFM, slfmMap=slfmMap, axis=1)
print("Found {0} Previous SetListFM Values".format(mmeDF["SetListFM"].count()))
print("Found {0} New SetListFM Values".format(mmeDF["SetListFMNew"].count()))

In [ ]:
mmeDF = mmeDF.drop(["SetListFM"], axis=1)
mmeDF = mmeDF.rename(columns={"SetListFMNew": "SetListFM"})

In [ ]:
pdbio.saveData(mmeDF)

In [ ]:
saData = mio.data.getSearchArtistData()
saData['MB'] = saData['mbid'].apply(ids.getmbid)
slfmMap = saData["id"]

In [ ]:
slfmMap

In [ ]:
saData

In [ ]:
saData["Refs"]['7bd4d270']

# Beatport From MusicBrainz

In [ ]:
from gate import MusicDBGate
gate  = MusicDBGate()
mdbio = gate.getIO("MusicBrainz")
bpio  = gate.getIO("Beatport")
bpMap = {}
mbMap = {}
for modVal in range(100):
    modValData = mdbio.data.getModValData(modVal)
    for mbid,mbidData in modValData.iteritems():
        if mbidData.profile.external is None:
            continue
        for item in mbidData.profile.external.get("Beatport", []):
            ref  = item.href if item is not None else None
            bpid = bpio.getdbid(ref) if isinstance(ref,str) else None
            if bpMap.get(mbid) is None:
                bpMap[mbid] = {}
            bpMap[mbid][(bpid,ref)] = mbidData.artist.name
    if modVal % 5 == 0:
        print(modVal,'\t',len(bpMap),'\t',len(mbMap))
bpMap = Series(bpMap)

In [ ]:
from pandas import Series,DataFrame
s = Series(bpMap)
beatportMap = {}
for k,v in bpMap[bpMap.map(len) == 1].iteritems():
    for k2,v2 in v.items():
        beatportMap[k] = k2[0]

for k,v in bpMap[bpMap.map(len) > 1].iteritems():
    for k2,v2 in v.items():
        print("beatportMap['{0}'] = {1: <10}  ## {2}  /  {3}".format(k,"'{0}'".format(k2[0]),k2[1],v2))
    print("")

In [ ]:
s = Series(beatportMap)
s.name = "Beatport"
df = DataFrame(s).join(mdbio.data.getSummaryNameData())

In [ ]:
externalsToGet = df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])]
from ioutils import FileIO
io = FileIO()
io.save(idata=externalsToGet, ifile="beatportMap.p")

In [ ]:
for mbid,row in df[~df.index.isin(pdbio.getNotNaDBIDs("MusicBrainz")["MusicBrainz"])].iterrows():
    artistName = row["Name"]
    bpid = row["Beatport"]
    pdbio.newArtist(name=artistName, Beatport=bpid, MusicBrainz=mbid)

In [ ]:
toadd = externalsToGet.reset_index()
toadd = toadd.rename(columns={"index": "MusicBrainz", "Name": "ArtistName"})

In [ ]:
for i in range(toadd.shape[0]):

In [ ]:
from uuid import uuid4
idx = []
for i in range(toadd.shape[0]):
    idx.append(str(uuid4()))
toadd.index = idx

In [ ]:
from pandas import concat
pdbio.saveData(concat([mmeDF,toadd]))

In [ ]:
mmeDF["Beatport"] = mmeDF["MusicBrainz"].map(beatportMap)

In [ ]:
pdbio.saveData(mmeDF)

In [ ]:
mdioData

# Other

In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()
pdbNames = mmeDF["ArtistName"]

In [ ]:
from match import MatchCounts, MatchDBCounts
db = "Discogs"
unMatched = MatchDBCounts(db).getUnmatched()
dbNames   = unMatched.head(5)["ArtistName"]

In [ ]:
mmeDF[mmeDF["ArtistName"].str.contains("m McLean")]

In [ ]:
for artistID,name in dbNames.iteritems():
    print("")
    print("#="*150)
    match = MatchString(base=name, compare=pdbNames).get(0.9)
    print("pdbio.newArtist('{0}', {1}='{2}')".format(name, db, artistID))
    print("pdbio.setdbid('', '{0}', '{1}')".format(db, artistID))
    if len(match) > 0:
        pdbData = pdbio.getRows(match.keys()).drop(['SetListFM', 'AlbumOfTheYear', 'Deezer', 'LastFM'], axis=1).to_string(index=True)
        print(pdbData)
    print("\n\n")

In [ ]:
#======================================================================================================================================================
pdbio.newArtist('Wiener Volksopernorchester', Discogs='833376', SetListFM='7bde12d0', Spotify='1gCaCu90v4PYeYxWq0joIW', RateYourMusic='308850', MusicBrainz='10145681830259485259250037629476827214')
pdbio.newArtist('Radio-Symphonie-Orchester Berlin', Discogs='688716', MusicBrainz='189002019997078465049180482273862734719', AllMusic='0002221657', Spotify='49TgMBH68KIFiOmLMoUOWY')
pdbio.newArtist('Staatskapelle Berlin', Discogs='833446', MusicBrainz='133018627337105403763302852079106379613', AllMusic="0002059537", Spotify="7vEPPI71V8dEHtEhPMAxWT", RateYourMusic="336903")
#======================================================================================================================================================
pdbio.newArtist("The King's College Choir Of Cambridge", Discogs='700443', MusicBrainz='273338754289470679440863230907769944743', AllMusic='0000608992', Spotify='0f3PsS9IQ6whvNMFFKnpjl', RateYourMusic='44555')
pdbio.newArtist('Hugo Winterhalter Orchestra', Discogs='395324', MusicBrainz='168341810660794848268304798831052693483', Spotify='5WVQxyLSWuV7XpjDlgNY53', RateYourMusic='55184')
pdbio.saveData()

In [ ]:
pdbio.saveData()

In [ ]:
pdbio.newArtist('Adventure Archives', Genius='2429680', Spotify='')
pdbio.newArtist('Home Runs of the Day', Genius='59781', Spotify='')
pdbio.newArtist('Rudy van Gelder', Genius='641939', Spotify='')
pdbio.newArtist('Chris Lord-Alge', Genius='108681', Spotify='')
pdbio.newArtist('Phil Elverum', Genius='666880', Spotify='')
pdbio.newArtist('La Vendicion', Genius='1194294', Spotify='')
pdbio.newArtist('Todd Tobias', Genius='28780', Spotify='')
pdbio.newArtist('EMI Music Publishing', Genius='648041', Spotify='')
pdbio.newArtist('Danja', Genius='10461', Spotify='')
pdbio.newArtist('Peter Asher', Genius='40509', Spotify='')
pdbio.newArtist('Raphael (Spanish singer)', Genius='1259329', Spotify='')
pdbio.newArtist('Creed Taylor', Genius='577179', Spotify='')
pdbio.newArtist('Ori Shochat - אורי שוחט', Genius='573979', Spotify='')
pdbio.newArtist('Dave Kutch', Genius='641341', Spotify='')
pdbio.newArtist('Tony Visconti', Genius='37268', Spotify='')
pdbio.newArtist('Honiro Label', Genius='1158245', Spotify='')
pdbio.newArtist('Chris Thomas', Genius='45700', Spotify='')
pdbio.newArtist('Benmont Tench', Genius='497398', Spotify='')
pdbio.newArtist('Tricky Stewart', Genius='27603', Spotify='')
pdbio.saveData()

In [ ]:
pdbio.newArtist('Rick Bonadio', Genius='192504', Spotify='2CZ8dMcFFZ1UYj52mUSaE6', Discogs='252040', RateYourMusic='1016711')
pdbio.newArtist('Nigel Godrich', Genius='38556', Spotify='0g7gHEXKEHU4snTwOZSxNO', AllMusic='0000869688', RateYourMusic='427029', Discogs='169094')
pdbio.newArtist('DooMee', Genius='1105756', Spotify='0Rb1j4P056IYg6ncXqslRr', RateYourMusic='1521674')
pdbio.newArtist('Mike Green', Genius='29122', Spotify='3kK0N0qZ5yuwWsqtaOfcQm', AllMusic='0000221368', RateYourMusic='1228102')
pdbio.newArtist('Prodigy', Genius='122', Spotify='1GwxXgEc6oxCKQ5wykWXFs', MusicBrainz='235890702021636578201118651139050295121', Discogs='134136', AllMusic="0000855953", RateYourMusic="27449")
pdbio.newArtist('Wheezy', Genius='339721', Spotify='4Ufo9whpMn1BwjnB3MJSYL', AllMusic='0002309613', RateYourMusic='1137548')
pdbio.newArtist('Benihana Boy', Genius='2361969', Spotify='0rygaaiuUiGN9W1o0mH6HV', AllMusic='0003796658', Discogs='7433745')
pdbio.newArtist('Mike McCready', Genius='372621', Spotify='7njqqUBXHc5fpyXmUlfOUL', AllMusic='0000491447', MusicBrainz='44313753520174712438202403431622254380', RateYourMusic="748912", Discogs='275980')
pdbio.saveData()

In [ ]:
#pdbio.getmbid()

In [ ]:
pdbio.saveData()